# CredibleX Growth Intelligence Engine
## 01 · Data Loading & Cleaning

This notebook loads the two synthetic datasets (`sme_data.csv`, `partner_data.csv`),
validates their structure, checks for data-quality issues, and prepares clean
versions for downstream analysis.

**Note:** All data in this project is synthetically generated for portfolio/demo
purposes. It does not represent real CredibleX customers or partners.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

sme = pd.read_csv('../data/sme_data.csv')
partner = pd.read_csv('../data/partner_data.csv')

print(f"SME dataset:     {sme.shape[0]:,} rows x {sme.shape[1]} columns")
print(f"Partner dataset: {partner.shape[0]:,} rows x {partner.shape[1]} columns")

SME dataset:     1,000 rows x 13 columns
Partner dataset: 50 rows x 9 columns


### 1.1 Schema & data types

In [2]:
sme.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SME_ID                 1000 non-null   str    
 1   Industry               1000 non-null   str    
 2   Emirate                1000 non-null   str    
 3   Business_Age           1000 non-null   float64
 4   Monthly_Revenue        1000 non-null   int64  
 5   Revenue_Growth         1000 non-null   float64
 6   Revenue_Volatility     1000 non-null   float64
 7   Transaction_Volume     1000 non-null   int64  
 8   Receivable_Days        1000 non-null   int64  
 9   Payable_Days           1000 non-null   int64  
 10  Cash_Buffer            1000 non-null   int64  
 11  Financing_Requirement  1000 non-null   int64  
 12  Repayment_Behaviour    1000 non-null   str    
dtypes: float64(3), int64(6), str(4)
memory usage: 101.7 KB


In [3]:
partner.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Partner_ID              50 non-null     str    
 1   Partner_Type            50 non-null     str    
 2   SME_Reach               50 non-null     int64  
 3   Transaction_Volume      50 non-null     int64  
 4   Industry_Focus          50 non-null     str    
 5   Financing_Relevance     50 non-null     float64
 6   Digital_Maturity        50 non-null     float64
 7   Growth_Rate             50 non-null     float64
 8   Integration_Complexity  50 non-null     str    
dtypes: float64(3), int64(2), str(4)
memory usage: 3.6 KB


### 1.2 Missing values

In [4]:
print("SME missing values:\n", sme.isna().sum().sum())
print("Partner missing values:\n", partner.isna().sum().sum())

SME missing values:
 0
Partner missing values:
 0


### 1.3 Duplicate ID check

In [5]:
print("Duplicate SME_IDs:", sme['SME_ID'].duplicated().sum())
print("Duplicate Partner_IDs:", partner['Partner_ID'].duplicated().sum())

Duplicate SME_IDs: 0
Duplicate Partner_IDs: 0


### 1.4 Range / sanity checks
Flag rows with implausible values (negative revenue, receivable days > 365, etc.).

In [6]:
issues = []
issues.append(('Negative Monthly_Revenue', (sme['Monthly_Revenue'] <= 0).sum()))
issues.append(('Receivable_Days > 365', (sme['Receivable_Days'] > 365).sum()))
issues.append(('Payable_Days > 365', (sme['Payable_Days'] > 365).sum()))
issues.append(('Business_Age <= 0', (sme['Business_Age'] <= 0).sum()))
issues.append(('Negative Financing_Requirement', (sme['Financing_Requirement'] <= 0).sum()))
issues.append(('Negative Partner SME_Reach', (partner['SME_Reach'] <= 0).sum()))

pd.DataFrame(issues, columns=['Check', 'Rows Flagged'])

,Check,Rows Flagged
0,Negative Monthly_Revenue,0
1,Receivable_Days > 365,0
2,Payable_Days > 365,0
3,Business_Age <= 0,0
4,Negative Financing_Requirement,0
5,Negative Partner SME_Reach,0


No data-quality issues were found — the synthetic generator constrains all fields to realistic ranges. In a production setting this notebook would also cross-check against a source-of-truth (core banking / CRM export) before promoting data downstream.

### 1.5 Categorical value counts (spot-check)

In [7]:
display(sme['Industry'].value_counts())
display(sme['Emirate'].value_counts())
display(sme['Repayment_Behaviour'].value_counts())

Industry
Retail                   176
F&B                      122
E-commerce               121
Professional Services    106
Trading                   95
Logistics                 94
Manufacturing             83
Construction              74
Healthcare                73
Hospitality               56
Name: count, dtype: int64

Emirate
Dubai             411
Abu Dhabi         224
Sharjah           131
Ajman              72
Ras Al Khaimah     69
Fujairah           54
Umm Al Quwain      39
Name: count, dtype: int64

Repayment_Behaviour
Good         371
Excellent    266
Fair         260
Poor         103
Name: count, dtype: int64

In [8]:
display(partner['Partner_Type'].value_counts())
display(partner['Integration_Complexity'].value_counts())

Partner_Type
Bank / NBFC Referral        8
Logistics Aggregator        7
POS / Payment Gateway       6
Supplier Network            5
B2B Marketplace             5
Telecom Provider            5
ERP / Invoicing Platform    5
Accounting Software         4
Free Zone Authority         3
E-commerce Platform         2
Name: count, dtype: int64

Integration_Complexity
Low       23
Medium    16
High      11
Name: count, dtype: int64

### 1.6 Descriptive statistics

In [9]:
sme.describe().T

,count,mean,std,min,25%,50%,75%,max
Business_Age,1000.0,5.932100,3.998178,0.50,3.00000,5.2000,7.80000,25.000
Monthly_Revenue,1000.0,180096.949000,201981.170778,18000.00,67687.75000,121435.5000,218296.75000,2500000.000
Revenue_Growth,1000.0,0.092884,0.216164,-0.35,-0.05925,0.0935,0.23925,0.650
Revenue_Volatility,1000.0,0.246942,0.139972,0.02,0.14000,0.2235,0.33500,0.808
Transaction_Volume,1000.0,433.807000,506.042357,26.00,159.00000,287.5000,504.50000,5128.000
Receivable_Days,1000.0,44.955000,26.804728,2.00,26.00000,39.0000,57.00000,150.000
Payable_Days,1000.0,31.161000,19.291170,3.00,17.00000,27.0000,40.00000,120.000
Cash_Buffer,1000.0,30.688000,20.546309,1.00,15.00000,26.0000,41.00000,120.000
Financing_Requirement,1000.0,194718.521000,221843.381219,10000.00,71952.00000,129445.5000,227438.00000,2394640.000


In [10]:
partner.describe().T

,count,mean,std,min,25%,50%,75%,max
SME_Reach,50.0,3.128220e+03,6.001792e+03,50.0,5.555000e+02,1201.50,3.626250e+03,4.091400e+04
Transaction_Volume,50.0,1.549832e+07,3.507786e+07,313205.0,2.861141e+06,5919922.00,1.621894e+07,2.415058e+08
Financing_Relevance,50.0,4.532000e+01,2.360865e+01,5.0,2.675000e+01,45.30,6.350000e+01,9.450000e+01
Digital_Maturity,50.0,6.066400e+01,2.079581e+01,13.7,4.940000e+01,61.85,7.762500e+01,9.730000e+01
Growth_Rate,50.0,1.737400e-01,2.204888e-01,-0.2,2.275000e-02,0.17,3.197500e-01,6.160000e-01


### 1.7 Save cleaned datasets
Since no quality issues were found, the cleaned datasets are the same as the
source files — saved here as explicit checkpoints for the next notebook.

In [11]:
sme.to_csv('../data/sme_data_clean.csv', index=False)
partner.to_csv('../data/partner_data_clean.csv', index=False)
print("Saved: data/sme_data_clean.csv, data/partner_data_clean.csv")

Saved: data/sme_data_clean.csv, data/partner_data_clean.csv


**Next:** `opportunity_engine.ipynb` — SME opportunity scoring, product matching, partner scoring, and early-warning classification.